# LangSmith Fundamentals — Observability for AI

This notebook walks through LangSmith's core features. **Keep https://smith.langchain.com open** in another tab — you'll check the UI after each demo.

**What you'll learn:**
1. Automatic tracing — zero-code observability for LangChain
2. `@traceable` — custom spans for your own functions
3. Agent tracing — seeing the full ReAct loop
4. Evaluation datasets — golden test sets
5. LLM-as-judge — automated quality scoring
6. Prompt management — versioning and deploying prompts

### Why observability matters

Traditional software: read the code to understand behavior.  
AI systems: the logic lives inside an LLM that reasons non-deterministically.  

Without observability, debugging is "add print statements and hope."  
With LangSmith, you get a full trace of every LLM call, tool invocation, and decision.

## Setup

Make sure your `.env` has:
```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_PROJECT=layer4-workshop
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

# Verify LangSmith is configured
print(f"LANGSMITH_TRACING: {os.environ.get('LANGSMITH_TRACING')}")
print(f"LANGSMITH_PROJECT: {os.environ.get('LANGSMITH_PROJECT')}")
print(f"API key set: {'Yes' if os.environ.get('LANGSMITH_API_KEY') else 'NO — fix your .env!'}")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

---
## 1. Automatic Tracing — Zero Extra Code

When `LANGSMITH_TRACING=true`, every LangChain call is traced automatically.

**What gets captured:**
- Input messages (prompt)
- Output (LLM response)
- Token counts (input, output, total)
- Latency
- Model name and parameters

No code changes needed — just set the env vars.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in one sentence."),
    ("user", "{question}"),
])

chain = prompt | llm | StrOutputParser()
result = chain.invoke({"question": "What is LangSmith?"})

print(f"Answer: {result}")
print()
print("Now go to: https://smith.langchain.com")
print("Select project: layer4-workshop")
print("You should see a trace with: ChatPromptTemplate → ChatOpenAI → StrOutputParser")

---
## 2. `@traceable` — Custom Spans for Your Own Functions

The `@traceable` decorator wraps any Python function as a **span** in the LangSmith trace tree.

**Key features:**
- Nested `@traceable` functions create parent → child spans automatically
- Add `metadata={...}` for filtering in the UI
- Add `run_type="chain"` or `"llm"` to categorize spans
- Inputs and outputs are captured automatically

**When to use:**
- Business logic that isn't a LangChain call
- Multi-step pipelines where you want to see each step
- Any function you want timing/debugging info for

In [ ]:
from langsmith import traceable

@traceable
def fetch_joke(topic: str) -> str:
    """This function becomes a span in LangSmith."""
    response = llm.invoke([
        ("system", "Tell a short one-liner joke."),
        ("user", f"Topic: {topic}"),
    ])
    return response.content

@traceable(metadata={"demo": "nested-spans", "exercise": "03"})
def joke_pipeline(topic: str) -> dict:
    """Parent span — fetch_joke is a child span inside it."""
    joke = fetch_joke(topic)
    rating = llm.invoke([
        ("system", "Rate this joke from 1-10. Reply with just the number."),
        ("user", joke),
    ])
    return {"joke": joke, "rating": rating.content.strip()}

result = joke_pipeline("software engineers")
print(f"Joke:   {result['joke']}")
print(f"Rating: {result['rating']}/10")
print()
print("In LangSmith you should see nested spans:")
print("  joke_pipeline")
print("    ├── fetch_joke")
print("    │     └── ChatOpenAI")
print("    └── ChatOpenAI (rating)")

---
## 3. Agent Tracing — Seeing the ReAct Loop

Agents involve multiple LLM calls and tool executions in a loop. LangSmith traces all of it automatically — no extra code.

**What you'll see in the trace:**
- Each iteration of the loop
- The LLM's reasoning (which tool to call and why)
- Tool inputs and outputs
- Total tokens across all iterations
- End-to-end latency

In [ ]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

tavily = TavilySearch(max_results=2)

@tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    response = tavily.invoke(query)
    return "\n".join(f"- {r['content'][:200]}" for r in response["results"])

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '2 + 2'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

agent = create_agent(
    llm,
    [search_web, calculate],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

result = agent.invoke({
    "messages": [("user", "What is 42 * 58? Also search for who invented Python.")]
})

print(result["messages"][-1].content[:400])
print()
print("In LangSmith → look for the agent trace.")
print("Expand it to see each ReAct iteration, tool call, and LLM reasoning step.")

---
## 4. Evaluation Datasets — Golden Test Sets

An evaluation dataset is a collection of **input → expected output** pairs. You use it to measure whether your agent/chain gives correct answers.

**LangSmith `Client` methods:**

| Method | Purpose |
|---|---|
| `client.create_dataset(name, description)` | Create a new dataset |
| `client.create_examples(dataset_id, examples)` | Add examples to a dataset |
| `client.list_datasets()` | List all datasets |
| `client.evaluate(target, data, evaluators)` | Run evaluation |

In [ ]:
import uuid
from langsmith import Client

client = Client()

dataset_name = f"demo-eval-{uuid.uuid4().hex[:6]}"
dataset = client.create_dataset(dataset_name, description="Quick demo dataset")

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is the capital of Japan?"},
            "outputs": {"answer": "Tokyo"},
        },
        {
            "inputs": {"question": "What is 7 * 8?"},
            "outputs": {"answer": "56"},
        },
        {
            "inputs": {"question": "Who wrote Romeo and Juliet?"},
            "outputs": {"answer": "William Shakespeare"},
        },
    ],
)

print(f"Created dataset: {dataset_name} (3 examples)")
print("View at: https://smith.langchain.com → Datasets")

---
## 5. LLM-as-Judge — Automated Quality Scoring

Traditional `assert output == expected` doesn't work for LLM outputs (too much variation). Instead, use an LLM to **judge** whether the answer is correct.

**Evaluation workflow:**
```
Dataset (inputs + expected outputs)
        │
        ▼
  Target function (your chain/agent)
        │
        ▼
  Evaluator (LLM-as-judge compares actual vs expected)
        │
        ▼
  Scores per example (visible in LangSmith Experiments)
```

**`client.evaluate()` parameters:**

| Parameter | Purpose |
|---|---|
| `target` | Function to evaluate: `inputs → outputs` |
| `data` | Dataset name (string) |
| `evaluators` | List of scoring functions |
| `experiment_prefix` | Name for the experiment run |
| `max_concurrency` | Parallel evaluation threads |

In [ ]:
# The target function — this is what we're evaluating
def answer_question(inputs: dict) -> dict:
    chain = (
        ChatPromptTemplate.from_messages([
            ("system", "Answer concisely in one sentence."),
            ("user", "{question}"),
        ])
        | llm
        | StrOutputParser()
    )
    return {"answer": chain.invoke({"question": inputs["question"]})}

# A simple evaluator — uses an LLM to judge correctness
def correctness_judge(inputs, outputs, reference_outputs):
    judge_response = llm.invoke([
        (
            "system",
            "Compare the actual answer to the expected answer. "
            "Reply with ONLY 'correct' or 'incorrect'.",
        ),
        (
            "user",
            f"Question: {inputs['question']}\n"
            f"Expected: {reference_outputs['answer']}\n"
            f"Actual: {outputs['answer']}",
        ),
    ])
    is_correct = "correct" in judge_response.content.lower().split()[0]
    return {"key": "correctness", "score": 1.0 if is_correct else 0.0}

# Run the evaluation
results = client.evaluate(
    answer_question,
    data=dataset_name,
    evaluators=[correctness_judge],
    experiment_prefix="demo-eval",
    max_concurrency=2,
)

print(f"\nExperiment: {results.experiment_name}")
print("View scores at: https://smith.langchain.com → Datasets → Experiments")

---
## 6. Prompt Management — Versioning and Deploying Prompts

LangSmith can store and version prompts. This lets you:
- **Push** prompts with version tags (`dev`, `staging`, `prod`)
- **Pull** prompts in code — swap prompts without deploying new code
- **Compare** versions side-by-side with evaluation results

**Key methods:**

| Method | Purpose |
|---|---|
| `client.push_prompt(name, object=prompt)` | Upload a prompt (creates or updates) |
| `client.pull_prompt(name)` | Download a prompt for use in code |
| `client.pull_prompt("name:tag")` | Pull a specific version by tag |

In [ ]:
# Version 1: Basic prompt
prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that specializes in {domain}."),
    ("user", "{question}"),
])

prompt_name = f"workshop-qa-{uuid.uuid4().hex[:6]}"

url = client.push_prompt(prompt_name, object=prompt_v1)
print(f"Pushed v1: {url}")

# Pull it back and use it
pulled = client.pull_prompt(prompt_name)
chain_v1 = pulled | llm | StrOutputParser()
result = chain_v1.invoke({"domain": "science", "question": "What is DNA?"})
print(f"v1 answer: {result[:120]}...")

In [ ]:
# Version 2: Enhanced prompt with more specific instructions
prompt_v2 = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in {domain}. Give detailed, accurate answers "
        "with concrete examples. Use bullet points for lists.",
    ),
    ("user", "{question}"),
])

client.push_prompt(prompt_name, object=prompt_v2)
print("Pushed v2 (same name, new version)")

pulled_v2 = client.pull_prompt(prompt_name)
chain_v2 = pulled_v2 | llm | StrOutputParser()
result = chain_v2.invoke({"domain": "science", "question": "What is DNA?"})
print(f"v2 answer: {result[:200]}...")
print()
print(f"Compare both versions: LangSmith → Prompts → {prompt_name}")

---
## Summary — LangSmith Features

```
Tracing (automatic)   →  Set LANGSMITH_TRACING=true, all LangChain calls traced
@traceable            →  Wrap any function to see it in the trace tree
Datasets              →  Golden test sets with inputs + expected outputs
Evaluators            →  LLM-as-judge scoring (correctness, helpfulness, etc.)
client.evaluate()     →  Run target function against dataset with evaluators
Prompt management     →  push_prompt / pull_prompt for versioning
```

**Key insight:** Treat prompts like code — version them, test them against eval datasets, and deploy specific versions. LangSmith makes this workflow practical.

**Next up:** Open `starter_tracing.py` and `starter_eval.py` to build your own tracing pipelines and evaluation suites.